# Denoising 2: IPCA Residuals → GDBN

**IPCA** (Instrumented PCA; Kelly, Pruitt & Su, 2019, *Journal of Financial Economics*) is PCA where each stock's **exposures to the factors depend on its characteristics** (size, value, momentum, volatility, news, ...). In plain PCA (notebook 14), Apple's loadings are fixed numbers learned from its past; in IPCA they come from what Apple **looks like today**.

$$r_{i,t} = \underbrace{z_{i,t-1}' \,\Gamma}_{\text{exposures } \beta_{i,t}}\; f_t \;+\; \underbrace{\varepsilon_{i,t}}_{\text{residual}}$$

- $z_{i,t-1}$: stock $i$'s 25 characteristics from the feature table at **yesterday's close** (ranked across stocks to $[-0.5, 0.5]$), plus a constant.
- $\Gamma$: a 26 × K table, learned from the past 5 years: how characteristics turn into exposures. It is the same for every stock, so it has few parameters.
- $f_t$: today's K factor returns.
- $\varepsilon_{i,t}$: the residual, the GDBN's new input.

```
yesterday's characteristics (26) --Gamma--> exposures (K)  \
                                                            >-- shared part;  return - shared part = RESIDUAL -> GDBN
today's returns ----cross-sectional regression--> factors (K) /
```

**Why it can beat PCA:** every stock gets exposures (even a new one with little history), and exposures change as the stock changes. Fitting $\Gamma$ uses alternating least squares (fix $\Gamma$, solve the factors; fix the factors, solve $\Gamma$; repeat).

| | Setting |
|---|---|
| Factors | **K = 5** (same as PCA) |
| Fit | each year, on the previous 5 years |
| Characteristics | all 25 features of the feature table, ranked daily, from day $t-1$ |

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import time
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pit_common as pc
import denoise_common as dc

D = pc.load()                        # returns and universe for the GDBN (everything after 2022 blanked)
cube = dc.load_cube()                # the feature table (features_daily.csv.gz) as arrays
GDBN_MASK = dc.gdbn_mask(D, cube)    # the GDBN's 52-stock universe, for R^2 on its own stocks


def frame(arr, rows, cols=None):
    return pd.DataFrame(arr, index=cube.days[rows], columns=cube.perm if cols is None else cols)

## 1. Characteristics and daily building blocks

In [ ]:
K, MAX_IT = 5, 100
Z = dc.rank_chars(cube)                                    # day x stock x 25, in [-0.5, 0.5]
L = Z.shape[2] + 1                                         # + constant
nd = len(cube.days)
ZZ, Zr, rr = np.zeros((nd, L, L)), np.zeros((nd, L)), np.zeros(nd)
for d in range(1, nd):
    m = cube.AVAIL[d]
    Zd = np.c_[Z[d - 1, m], np.ones(m.sum())]              # yesterday's characteristics
    r = cube.R[d, m].astype(float)
    ZZ[d], Zr[d], rr[d] = Zd.T @ Zd, Zd.T @ r, r @ r
print(f"{nd:,} days of building blocks; L = {L} characteristics (incl. constant)")

## 2. Fit Γ each year (alternating least squares) and compute residuals

In [ ]:
def factors_given(G, ZZ_, Zr_):
    A = np.einsum("lk,tlm,mj->tkj", G, ZZ_, G, optimize=True)
    b = Zr_ @ G
    return np.linalg.solve(A + 1e-10 * np.eye(K), b[..., None])[..., 0], A, b


def fit_ipca(fd):
    U_, _, _ = np.linalg.svd(Zr[fd].T, full_matrices=False)
    G, prev = U_[:, :K], -np.inf                           # start: PCA of the characteristic-managed portfolios
    for it in range(MAX_IT):
        f, _, _ = factors_given(G, ZZ[fd], Zr[fd])
        den = np.einsum("tk,tj,tlm->kljm", f, f, ZZ[fd], optimize=True).reshape(K * L, K * L)
        num = np.einsum("tk,tl->kl", f, Zr[fd], optimize=True).reshape(K * L)
        G = np.linalg.solve(den + 1e-10 * np.eye(K * L), num).reshape(K, L).T
        G = np.linalg.svd(G, full_matrices=False)[0]       # keep the scale fixed (only G x f matters)
        f, A, b = factors_given(G, ZZ[fd], Zr[fd])
        ssr = rr[fd] - 2 * (f * b).sum(axis=1) + np.einsum("tk,tkj,tj->t", f, A, f)
        fit = 1 - ssr.sum() / rr[fd].sum()
        if fit - prev < 1e-7:
            break
        prev = fit
    return G, fit, it + 1


by_year, rows, gammas = {}, [], {}
t0 = time.time()
for Y in dc.YEARS:
    fd, ad = dc.fit_days(cube, Y), dc.apply_days(cube, Y)
    G, fit, iters = fit_ipca(fd)
    gammas[Y] = G
    f_all, _, _ = factors_given(G, ZZ[ad], Zr[ad])
    resid = np.full((len(ad), len(cube.perm)), np.nan, dtype=np.float32)
    for n, d in enumerate(ad):
        m = cube.AVAIL[d]
        Zd = np.c_[Z[d - 1, m], np.ones(m.sum())]
        resid[n, m] = cube.R[d, m] - Zd @ G @ f_all[n]
    by_year[Y] = SimpleNamespace(RET=frame(resid, ad), SENT=None, ATTN=None,
                                 FACT=pd.DataFrame(f_all, index=cube.days[ad], columns=[f"ipca{k + 1}" for k in range(K)]))
    yd = cube.days[ad].year == Y
    actual, fitted = cube.R[ad][yd], (cube.R[ad] - resid)[yd]
    rows.append({"year": Y, "ALS iterations": iters, "fit R2 (in sample)": fit,
                 "R2 S&P members (out of sample)": dc.r2(np.where(cube.MEM[ad][yd], actual, np.nan), fitted),
                 "R2 GDBN universe (out of sample)": dc.r2(np.where(GDBN_MASK[ad][yd], actual, np.nan), fitted)})
r2_table = pd.DataFrame(rows).set_index("year")
print(f"Done in {time.time() - t0:.0f}s")
r2_table.round(3)

**Reading it:** R² = share of each day's excess-return moves explained by the 5 IPCA factors, on days **not** used to fit Γ. Compare with PCA's R² (notebook 14): IPCA uses fewer parameters but more information (characteristics).

### What drives the factors? (Γ for the last fitted year)

Each column is one factor; each row shows how much a characteristic raises a stock's exposure to it. This is the **interpretable** part of IPCA: you can read what each factor is about.

In [ ]:
G = gammas[max(gammas)]
fig, ax = plt.subplots(figsize=(7, 8))
lim = np.abs(G).max()
im = ax.imshow(G, cmap="RdBu_r", vmin=-lim, vmax=lim, aspect="auto")
ax.set_yticks(range(L)); ax.set_yticklabels(dc.FEATS + ["constant"], fontsize=8)
ax.set_xticks(range(K)); ax.set_xticklabels([f"factor {k + 1}" for k in range(K)])
ax.set_title(f"Γ for {max(gammas)} (fit on the 5 years before)")
plt.colorbar(im, ax=ax, shrink=0.6); plt.tight_layout(); plt.show()

## GDBN on the denoised data

Four versions of the GDBN, all on the same 52-stock point-in-time universe, walk-forward, λ rule (best rank IC on 2006–2020), and 2021–2022 validation:

| Version | Inputs (yesterday's values) |
|---|---|
| **raw daily** | excess return, sentiment, attention, ETFs (notebook 06) |
| **raw slow** | past-week and past-month averages of the same (notebook 10, the best so far) |
| **IPCA daily** | the same, with the **denoised** series in place of the raw one (the IPCA residual return), plus the 5 IPCA factor returns as market nodes |
| **IPCA slow** | past-week and past-month averages of the denoised series, plus the factors' averages |

The target is unchanged: each stock's next-day excess return. The raw versions are saved the first time and loaded in later notebooks.

**Columns:** `arrows/quarter` = network size at the chosen λ; `arrow stability` = how much consecutive quarters' networks overlap (1 = identical). Denser networks overlap more just by chance, so `arrows @ raw λ` and `stability @ raw λ` repeat both at the **same λ as the raw GDBN** of the same kind: the fair, like-for-like comparison. Then rank IC, Newey–West t, gross Sharpe (before costs), and breakeven bps (the cost level that would wipe out the profit) for each period.

In [ ]:
runs = dc.raw_gdbn(D)
runs["IPCA daily"] = dc.walk_forward("ipca_daily", D, dc.gdbn_features(by_year, "daily"))
runs["IPCA slow"] = dc.walk_forward("ipca_slow", D, dc.gdbn_features(by_year, "slow"))
table, scores = dc.evaluate(D, runs)
print(table.round(4).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for name, s in scores.items():
    ic = pc.rank_ic(s, pc.forward_excess(D, s, 1))
    axes[0].plot(ic.cumsum(), label=name, ls="--" if name.startswith("raw") else "-")
for name, res in runs.items():
    k, _ = pc.choose_lambda(D, res, 1)
    axes[1].plot([q["r"] for q in res], [(q["W"][:, :, k] != 0).sum() for q in res], label=name,
                 ls="--" if name.startswith("raw") else "-")
axes[0].axvline(pd.Timestamp("2021-01-01"), color="gray", lw=1); axes[0].set_title("Cumulative rank IC (rising = skill; line = validation starts)")
axes[1].set_title("Arrows in each quarter's network (at each version's chosen λ)")
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

**Reading it:**
- **Did denoising help prediction?** Compare the IPCA rows with the raw rows of the same kind (daily vs daily, slow vs slow): rank IC, gross Sharpe, breakeven, and especially **2021–2022**.
- **Did it clean up the connections?** Compare **stability @ raw λ** (same λ, so the same price per arrow): higher means the network keeps finding the same links from quarter to quarter, which is what real (not noise-driven) connections should do. The plain `arrow stability` column is inflated for versions that chose a smaller λ (bigger networks).
- Notebook 18 puts all four denoisers side by side.

In [ ]:
dc.save("ipca", table, r2_table)